# Vision Transformer for Image Classification

## What this notebook demonstrates

- Vision Transformer architecture for image classification
- patch embeddings and positional embeddings
- self-attention and multi-head attention
- transformer encoder blocks
- PyTorch training and evaluation workflows

## Academic context

This notebook is a cleaned portfolio version of MSc coursework and implementation practice. It is included to demonstrate foundational computer vision and deep learning skills.


## GPU


If CUDA is available, the setup code uses it automatically; otherwise the notebook runs on CPU.


If CUDA is available, the setup code uses it automatically; otherwise the notebook runs on CPU.


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torch.utils.data import sampler

import torchvision.datasets as datasets
import torchvision.transforms as T

import numpy as np
import math
from tqdm import tqdm

USE_GPU = True
dtype = torch.float32 # We will be using float throughout this tutorial.

if USE_GPU and torch.cuda.is_available():
    device = torch.device('cuda')
else:
    device = torch.device('cpu')

# Constant to control how frequently we print train loss.
print_every = 100
print('using device:', device)

## Preparation and Visualization


### Load CIFAR-10 dataset


Now, let's load the CIFAR-10 dataset.

This might take a couple minutes the first time you do it, but the files should stay cached after that.

In previous parts of the earlier notebooks we had to write our own code to download the CIFAR-10 dataset, preprocess it, and iterate through it in minibatches.

To make the whole procedure more efficient, we managed to save the whole dataset in ``.npy`` files.

PyTorch provides some convenient tools to automate this process even more for us.


In [ ]:
import torch
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, random_split

# Define the mean and standard deviation for CIFAR-10
mean = (0.4914, 0.4822, 0.4465)  # Precomputed mean for CIFAR-10
std = (0.2023, 0.1994, 0.2010)   # Precomputed std for CIFAR-10

# Define the transform for preprocessing and data augmentation
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean, std)
])

# Load the CIFAR-10 dataset
#dataset_path = "data"  
dataset_path = "data" 
cifar10_dataset = datasets.CIFAR10(root=dataset_path, train=True, download=True, transform=transform) 

# Split the dataset into train, val, and test sets
num_train = 45000  # Number of training examples
num_val = 5000    # Number of validation examples
num_test = 10000   # Number of test examples (CIFAR-10 test set size)

train_dataset, val_dataset = random_split(cifar10_dataset, [num_train, num_val])

test_dataset = datasets.CIFAR10(root=dataset_path, train=False, download=True, transform=transform)

# Define DataLoaders
batch_size = 64

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

# Print dataset sizes for verification
print(f"Train set size: {len(train_dataset)}")
print(f"Validation set size: {len(val_dataset)}")
print(f"Test set size: {len(test_dataset)}")

### Visualize the class distributions


We have now defined our train, validation and test dataloaders. However, it would be good to check the data distribution of each dataloader, in order to check how many examples per class are included. We do this as a sanity check, as we want our dataloaders to be distributed close to uniform. Let's define some helper functions:


In [ ]:
import numpy as np
from collections import Counter

def get_class_distribution(loader, dataset_classes):
    """
    Computes the distribution of classes in a DataLoader.

    Args:
    - loader: DataLoader to inspect
    - dataset_classes: List of class names for the dataset

    Returns:
    - class_counts: Dictionary with class names as keys and counts as values
    """
    class_counts = Counter()
    for inputs, labels in loader:
        class_counts.update(labels.numpy())

    # Convert to a dictionary with class names
    class_distribution = {dataset_classes[i]: class_counts[i] for i in range(len(dataset_classes))}
    return class_distribution

def display_class_distribution(distribution, title="Class Distribution"):
    """
    Displays the class distribution as a bar chart.

    Args:
    - distribution: Dictionary with class names as keys and counts as values
    - title: Title for the bar chart
    """
    import matplotlib.pyplot as plt

    class_names = list(distribution.keys())
    counts = list(distribution.values())

    plt.figure(figsize=(10, 6))
    plt.bar(class_names, counts)
    plt.title(title)
    plt.xlabel("Class")
    plt.ylabel("Count")
    plt.xticks(rotation=45)
    plt.show()

Now, let's visualize the distribution of the examples per class for each dataloader:


In [ ]:
# CIFAR-10 classes
cifar10_classes = ['airplane', 'automobile', 'bird', 'cat', 'deer',
                   'dog', 'frog', 'horse', 'ship', 'truck']

# Compute and display class distribution for each DataLoader
for loader, name in [(train_loader, "Train"), (val_loader, "Validation"), (test_loader, "Test")]:
    distribution = get_class_distribution(loader, cifar10_classes)
    print(f"{name} DataLoader Distribution:")
    for cls, count in distribution.items():
        print(f"  {cls}: {count}")
    display_class_distribution(distribution, title=f"{name} DataLoader Class Distribution")

### Visualize CIFAR-10 images


Now let's visualize a few examples from each class in the CIFAR-10 dataset.

By displaying images alongside their corresponding class labels, we can verify the dataset’s contents and ensure the classes are represented correctly.

This visualization helps us understand the data distribution and provides insights into the dataset’s characteristics.


In [ ]:
import matplotlib.pyplot as plt

def visualize_images_per_class(dataset, dataset_classes, num_images=5):
    """
    Visualizes a few images from each class in the dataset.

    Args:
    - dataset: The dataset to sample from.
    - dataset_classes: List of class names for the dataset.
    - num_images: Number of images to display per class.
    """
    class_to_indices = {cls: [] for cls in range(len(dataset_classes))}

    # Group indices by class
    for idx, (_, label) in enumerate(dataset):
        if len(class_to_indices[label]) < num_images:
            class_to_indices[label].append(idx)
        if all(len(indices) == num_images for indices in class_to_indices.values()):
            break

    # Plot the images
    fig, axes = plt.subplots(len(dataset_classes), num_images + 1, figsize=((num_images + 1) * 2, len(dataset_classes) * 2))
    for cls_idx, cls_name in enumerate(dataset_classes):
        indices = class_to_indices[cls_idx]

        # Add class name to the first column of each row
        axes[cls_idx, 0].text(0.5, 0.5, cls_name, fontsize=12, ha='center', va='center', transform=axes[cls_idx, 0].transAxes)
        axes[cls_idx, 0].axis("off")

        for img_idx, data_idx in enumerate(indices):
            img, label = dataset[data_idx]
            ax = axes[cls_idx, img_idx + 1]  # Shift by 1 for the class name column
            img = img.permute(1, 2, 0)  # Convert CHW to HWC
            img = img * torch.tensor(std).view(1, 1, 3) + torch.tensor(mean).view(1, 1, 3)  # De-normalize
            img = torch.clamp(img, 0, 1)  # Clip values to valid range
            ax.imshow(img)
            ax.axis("off")
    plt.tight_layout()
    plt.show()

# Visualize 5 images per class for CIFAR-10 train set
visualize_images_per_class(cifar10_dataset, cifar10_classes, num_images=5)

## Introduction to Vision Transformers


### From Transformers to Vision Transformers


Transformers, originally introduced in natural language processing (NLP) through the groundbreaking [“Attention is All You Need”](https://arxiv.org/abs/1706.03762) paper, revolutionized the way sequential data is processed.

They replaced recurrent neural networks (RNNs) and convolutional neural networks (CNNs) in NLP with a mechanism based on *multi-head self-attention* (MHSA), enabling models to capture long-range dependencies and parallelize computation effectively.

This architecture laid the foundation for powerful language models like [BERT](https://arxiv.org/abs/1810.04805) and [GPT](https://arxiv.org/abs/2005.14165).

The success of transformers in NLP inspired researchers to explore their potential in computer vision tasks.

This gave rise to [Vision Transformers (ViTs)](https://arxiv.org/abs/2010.11929), a novel adaptation of the transformer architecture for image processing. Unlike traditional convolutional neural networks that operate *hierarchically* with *local receptive fields*, ViTs treat images as sequences of patches, akin to words in a sentence.

Each image is divided into fixed-size patches, flattened, and linearly embedded before being fed into the transformer model.


The key innovation of ViTs is their ability to model *global relationships* between image patches using *self-attention*, allowing them to capture *long-range dependencies* in visual data.

This differs from CNNs, which rely on *localized filters* and *hierarchical feature extraction*.

Vision Transformers, when pre-trained on large-scale datasets, have demonstrated competitive performance on a variety of vision tasks, including image classification, object detection, and segmentation.

While ViTs offer flexibility and scalability, they also pose challenges!

Their performance heavily depends on the *availability of large datasets for training*, as they lack the inductive biases inherent in CNNs (e.g., translation invariance).

This has spurred the development of hybrid models that combine CNNs’ inductive biases with the global modeling power of transformers.


### ConvNets vs. Vision Transformers


As stated, **Convolutional Neural Networks** (ConvNets or CNNs) have long been the dominant architecture for image classification tasks.

However, recent advancements in deep learning have introduced **Vision Transformers** (ViTs), which adapt the success of transformers in NLP to vision tasks.

Key Differences Between CNNs and Vision Transformers:

1.	**Feature Extraction**:

-	CNNs use convolutional filters to extract spatial features from images.

-	ViTs split an image into patches, flatten them, and process them as a sequence of tokens (like words in NLP).

2.	**Global Attention**:

-	CNNs capture local patterns through small filters.

-	ViTs use self-attention mechanisms, enabling them to model long-range dependencies across the entire image.

Vision Transformer Architecture:

1.	**Patch Embedding**: The image is divided into fixed-size patches, which are flattened and projected into a higher-dimensional space.

2.	**Transformer Encoder**:
-	*Multi-Head Self-Attention*: Captures dependencies between image patches.
-	*Feedforward Neural Network*: Processes attention outputs.
-	*Residual Connections* and *Layer Normalization* stabilize training.

3.	**Classification Head**: A special `[CLS]` token is passed through the transformer layers and then projected to class logits.


## Custom Vision Transformer with Pytorch Module API


To start with, we will implement a custom **Vision Transformer** using the `Module API`. The model architecture will include:

1.	**Patch Embedding**:

    - The input image will be split into smaller non-overlapping patches.

    - Each patch will be projected into a fixed-size embedding vector using a learnable linear projection.

2. **Position Embeddings**:

    - Learnable position embeddings will be added to each patch embedding to encode spatial information.

3. **`[CLS]` token**:

    - A learnable `[CLS]` token will be prepended to the sequence of patch embeddings, which will serve as a representation of the entire image for classification.

2.	**Transformer Encoder**:

- A stack of transformer blocks, each consisting of:

    -	Multi-head self-attention (MHSA).

    -	Feedforward neural network (FFN) / Multi-layer Perceptron (MLP).

    -	Skip connections and layer normalization.

3.	**Classification Head**: A linear classifier that takes the `[CLS]` token’s representation from the transformer encoder and projects it into class logits.


#### Configuration Parameters


Throughout the implementation below, you are going to need some configuration parameters. These are:

1. **Patch Size (`patch_size`)**: Determines the size of each patch. For an input image of size 32x32, a patch size of 4 results in a sequence of 8x8 = 64 patches.
2. **Hidden Size (`hidden_size`)**: The dimensionality of the embeddings and transformer layers.
3. **Number of Transformer Layers (`num_hidden_layers`)**: Specifies the number of transformer blocks in the encoder.
4. **Number of Attention Heads (`num_attention_heads`)**: Defines how many independent attention mechanisms are computed in parallel.
5. **Intermediate Size (`intermediate_size`)**: The size of the feed-forward network within each transformer block. Typically set to 4 times the `hidden_size`.
6. **Dropout Probabilities (`hidden_dropout_prob`, `attention_probs_dropout_prob`)**: Dropout rates for regularization during training.
7. **Image Size (`image_size`)**: The height and width of the input images (assumes square images).
8. **Number of Classes (`num_classes`)**: The number of output classes (e.g., 10 for CIFAR-10).
9. **Number of Channels (`num_channels`)**: The number of channels in the input images (e.g., 3 for RGB images).
10. **Query-Key-Value Bias (`qkv_bias`)**: Whether to include biases in the query, key, and value projections.
11. **Initializer Range (`initializer_range`)**: The standard deviation for weight initialization.

```python
# Configuration for custom ViT model
config = {
    "patch_size": 4,  # Input image size: 32x32 -> 8x8 patches
    "hidden_size": 48,
    "num_hidden_layers": 4,
    "num_attention_heads": 4,
    "intermediate_size": 4 * 48,  # 4 * hidden_size
    "hidden_dropout_prob": 0.0,
    "attention_probs_dropout_prob": 0.0,
    "initializer_range": 0.02,
    "image_size": 32,
    "num_classes": 10,  # CIFAR-10 number of classes
    "num_channels": 3,
    "qkv_bias": True,
    "use_faster_attention": True,
}
```


### Module API: Patch Embedding


In this step, you will implement the `PatchEmbedding` module, which converts an image into a sequence of flattened patches and projects each patch into a fixed-size embedding vector.+


#### Key Steps:


1. Use a convolutional layer (`nn.Conv2d()`) to split the image into non-overlapping patches and project each patch into a vector space of size `hidden_size`.
2. Flatten the spatial dimensions of the patches and rearrange them into a sequence of embeddings.
3. Return the sequence of patch embeddings.

The input to the `PatchEmbedding` module will be a batch of images of shape `(batch_size, num_channels, image_size, image_size)`.

The output will be a tensor of shape `(batch_size, num_patches, hidden_size)`.


#### Implementation


**Implementation note.**

Complete the `PatchEmbedding` module by:
1. Calculating the number of patches from the image size and patch size.
2. Defining the projection layer to split the image into patches and project them.
3. Completing the `forward` method to process the input image into patch embeddings.

Refer to the comments in the code snippet for guidance.


In [ ]:
class PatchEmbedding(nn.Module):
    """
    Converts an image into a sequence of flattened patches and projects them into a vector space.

    Parameters:
    - config: A dictionary containing the following keys:
        - "image_size" (int): The height and width of the input image (assumes square images).
        - "patch_size" (int): The height and width of each patch (assumes square patches).
        - "num_channels" (int): The number of input channels (e.g., 3 for RGB images).
        - "hidden_size" (int): The dimensionality of the output embedding for each patch.
    """

    def __init__(self, config):
        super().__init__()
        # Extract configuration parameters
        self.image_size = config["image_size"]
        self.patch_size = config["patch_size"]
        self.num_channels = config["num_channels"]
        self.hidden_size = config["hidden_size"]

        self.num_patches = int((self.image_size / self.patch_size)**2)

        # The projection layer splits the image into patches and projects them         #
        self.projection = torch.nn.Conv2d(
            in_channels=self.num_channels,
            out_channels=self.hidden_size,
            kernel_size=self.patch_size,
            stride=self.patch_size
        )

    def forward(self, x):
        """
        Forward pass:
        - Input: A batch of images with shape (batch_size, num_channels, image_size, image_size)
        - Output: A tensor of shape (batch_size, num_patches, hidden_size), where each patch is projected into a hidden vector space.
        """
        x = self.projection(x)

        x = x.flatten(2)
        x = x.transpose(1, 2)

        return x

### Module API: Embeddings


The `Embeddings` module combines patch embeddings with a learnable [CLS] token and positional embeddings. These embeddings form the input sequence for the transformer encoder.


#### Key Steps:


1. Generate patch embeddings for the input image using the `PatchEmbedding` module.
2. Add a learnable [CLS] token at the beginning of the sequence to represent the entire image.
3. Add learnable positional embeddings to encode spatial information for each patch and the [CLS] token.
4. Apply dropout to the final sequence for regularization.

The input to the `Embeddings` module will be a batch of images of shape `(batch_size, num_channels, image_size, image_size)`.

The output will be a tensor of shape `(batch_size, num_patches + 1, hidden_size)`, where `num_patches` depends on the image size and patch size.


#### Implementation


**Implementation note.**

Complete the `Embeddings` module by:
1. Initializing the [CLS] token and position embeddings.
2. The `forward` method to:
   - Generate patch embeddings.
   - Append the [CLS] token to the beginning of the sequence.
   - Add positional embeddings to the sequence.
   - Apply dropout to the final sequence.

Refer to the comments in the code snippet for guidance.


In [ ]:
class Embeddings(nn.Module):
    """
    Combines patch embeddings with the [CLS] token and position embeddings.

    This module performs the following steps:
    1. Generates patch embeddings for the input image.
    2. Adds a learnable [CLS] token at the beginning of the sequence.
    3. Adds positional embeddings to encode the spatial information.
    4. Applies dropout for regularization.

    Parameters:
    - config (dict): A dictionary containing the following keys:
        - "image_size" (int): The height and width of the input image (assumes square images).
        - "patch_size" (int): The height and width of each patch (assumes square patches).
        - "num_channels" (int): The number of input channels (e.g., 3 for RGB images).
        - "hidden_size" (int): The dimensionality of the patch embeddings and transformer embeddings.
        - "hidden_dropout_prob" (float): Dropout probability applied after adding positional embeddings.
    """

    def __init__(self, config):
        super().__init__()
        # Save configuration
        self.config = config
        self.patch_embeddings = PatchEmbedding(config)

        self.cls_token = nn.Parameter(torch.zeros(1, 1, config['hidden_size']))

        num_patches = int(self.patch_embeddings.num_patches)
        hidden_size = self.patch_embeddings.hidden_size
        self.positional_embeddings = nn.Parameter(torch.zeros(1, num_patches + 1, hidden_size))

        # Dropout for regularization
        self.dropout = nn.Dropout(config["hidden_dropout_prob"])

    def forward(self, x):
        """
        Forward pass:
        - Input: A batch of images of shape (batch_size, num_channels, image_size, image_size)
        - Output: A sequence of embeddings of shape (batch_size, num_patches + 1, hidden_size)
        """
        x = self.patch_embeddings(x)

        batch_size = x.size(0)
        cls_tokens = self.cls_token.expand(batch_size, -1, -1)

        x = torch.cat((cls_tokens, x), dim=1)

        x += self.positional_embeddings

        # Apply dropout
        x = self.dropout(x)

        return x

### Module API: Attention head


In this step, you will implement a single `AttentionHead` for the self-attention mechanism.

An attention head is a critical component of the transformer architecture, allowing the model to focus on different parts of the input sequence.


#### Key Steps:


1. Project the input tensor into **query**, **key**, and **value** vectors using linear layers.
2. Compute the **scaled dot-product attention**:
   
   $\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{Q \cdot K^T}{\sqrt{d_k}}\right) \cdot V$
   
3. Apply dropout to the attention probabilities for regularization.
4. Return:
   - **Attention Output**: A tensor of shape `(batch_size, sequence_length, attention_head_size)`.
   - **Attention Probabilities**: A tensor of shape `(batch_size, sequence_length, sequence_length)`.

The input to the `AttentionHead` module will be a tensor of shape `(batch_size, sequence_length, hidden_size)`.


#### Implementation


**Implementation note.**

Complete the `AttentionHead` module by:
1. Defining the linear layers to compute query, key, and value vectors.
2. The `forward` method to:
   - Compute the query, key, and value projections.
   - Calculate the attention scores.
   - Compute the attention probabilities.
   - Compute the attention output.

Refer to the comments in the code snippet for guidance.


In [ ]:
class AttentionHead(nn.Module):
    """
    Implements a single attention head for self-attention.

    This module performs the following steps:
    1. Projects the input into query, key, and value vectors.
    2. Computes scaled dot-product attention:
        - attention(Q, K, V) = softmax(Q * K^T / sqrt(d_k)) * V
    3. Applies dropout to the attention probabilities for regularization.

    Parameters:
    - hidden_size (int): Dimensionality of the input embeddings.
    - attention_head_size (int): Dimensionality of each attention head.
    - dropout (float): Dropout probability applied to attention probabilities.
    - bias (bool): Whether to include biases in the projection layers (default: True).
    """

    def __init__(self, hidden_size, attention_head_size, dropout, bias=True):
        super().__init__()
        # Save dimensions
        self.hidden_size = hidden_size
        self.attention_head_size = attention_head_size

        self.query = nn.Linear(hidden_size, attention_head_size, bias=bias)
        self.key = nn.Linear(hidden_size, attention_head_size, bias=bias)
        self.value = nn.Linear(hidden_size, attention_head_size, bias=bias)

        # Dropout for regularization
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        """
        Forward pass:
        - Input: A tensor of shape (batch_size, sequence_length, hidden_size)
        - Output:
          - attention_output: A tensor of shape (batch_size, sequence_length, attention_head_size)
          - attention_probs: The attention probabilities (batch_size, sequence_length, sequence_length)
        """
        # Compute query, key, and value vectors
        query = self.query(x)  # Shape: (batch_size, sequence_length, attention_head_size)
        key = self.key(x)      # Shape: (batch_size, sequence_length, attention_head_size)
        value = self.value(x)  # Shape: (batch_size, sequence_length, attention_head_size)

        scaling_factor = torch.sqrt(torch.tensor(self.attention_head_size, dtype=torch.float32))
        scores = torch.matmul(query, key.transpose(-1, -2)) / scaling_factor

        attention_probs = F.softmax(scores, dim=-1)

        # Apply dropout to attention probabilities
        attention_probs = self.dropout(attention_probs)

        attention_output = torch.matmul(attention_probs, value)

        return attention_output, attention_probs

### Module API: Multi-head self-attention (MHSA)


The `MultiHeadAttention` module implements the multi-head attention mechanism used in transformers.

Multi-head attention allows the model to focus on different parts of the input sequence simultaneously by splitting the attention computation across multiple attention heads.


#### Key Steps:
1. Compute attention for each head independently using the `AttentionHead` module.
2. Concatenate the outputs from all heads along the last dimension.
3. Project the concatenated outputs back to the hidden size using a linear layer.
4. Optionally return attention probabilities for visualization or analysis.

The input to the `MultiHeadAttention` module will be a tensor of shape `(batch_size, sequence_length, hidden_size)`. The output will include:
- **Attention Output**: A tensor of shape `(batch_size, sequence_length, hidden_size)`.
- **Attention Probabilities** (optional): A tensor of shape `(batch_size, num_heads, sequence_length, sequence_length)` if `output_attentions=True`.


#### Implementation

**Implementation note.**

Complete the `MultiHeadAttention` module by:
1. Defining the attention heads as a list of independent `AttentionHead` modules.
2. The `forward` method to:
   - Compute attention outputs for each head.
   - Concatenate the outputs from all heads.
   - Project the concatenated output back to the hidden size.
   - Optionally return attention probabilities.


In [ ]:
class MultiHeadAttention(nn.Module):
    """
    Implements the multi-head attention mechanism for transformers.

    Multi-head attention allows the model to jointly focus on different parts of the input sequence by splitting
    the attention computation across multiple attention heads. This module:
    1. Computes attention for each head independently using the `AttentionHead` module.
    2. Concatenates the outputs from all heads.
    3. Projects the concatenated output back to the hidden size dimension.

    Parameters:
    - config (dict): Configuration dictionary containing:
        - "hidden_size" (int): Dimensionality of the input and output embeddings.
        - "num_attention_heads" (int): Number of attention heads.
        - "qkv_bias" (bool): Whether to include biases in query, key, and value projections.
        - "attention_probs_dropout_prob" (float): Dropout probability for attention probabilities.
        - "hidden_dropout_prob" (float): Dropout probability for the output projection layer.
    """

    def __init__(self, config):
        super().__init__()

        # Extract configuration parameters
        self.hidden_size = config["hidden_size"]
        self.num_attention_heads = config["num_attention_heads"]
        self.qkv_bias = config["qkv_bias"]

        self.attention_head_size = int(self.hidden_size / self.num_attention_heads)
        assert (
            self.hidden_size % self.num_attention_heads == 0
        ), "hidden_size must be divisible by num_attention_heads."

        # Total size of all heads combined
        self.all_head_size = self.num_attention_heads * self.attention_head_size

        # Make use of a `nn.ModuleList()`, which will store inside the list of modules  #
        self.attention_heads = nn.ModuleList([AttentionHead(
            self.hidden_size,
            self.attention_head_size,
            config['attention_probs_dropout_prob'],
            bias=self.qkv_bias) for _ in range(self.num_attention_heads)])

        self.output_projection = nn.Linear(self.all_head_size, self.hidden_size)

        # Define dropout for the output projection
        self.output_dropout = nn.Dropout(config["hidden_dropout_prob"])

    def forward(self, x, output_attentions=False):
        """
        Forward pass for multi-head attention.

        Parameters:
        - x (Tensor): Input tensor of shape (batch_size, sequence_length, hidden_size).
        - output_attentions (bool): If True, returns attention probabilities for each head.

        Returns:
        - attention_output (Tensor): Output tensor of shape (batch_size, sequence_length, hidden_size).
        - attention_probs (Tensor, optional): Attention probabilities for each head of shape
          (batch_size, num_heads, sequence_length, sequence_length) if `output_attentions` is True.
        """
        attention_outputs = []  # contains tuples
        for head in self.attention_heads:
            attention_output, attention_probs = head(x)
            attention_outputs.append((attention_output, attention_probs))

        attention_output = torch.cat([output[0] for output in attention_outputs], dim=-1)

        attention_output = self.output_projection(attention_output)
        attention_output = self.output_dropout(attention_output)

        # Optionally return attention probabilities
        if output_attentions:
            attention_probs = torch.stack(
                [attention_probs for _, attention_probs in attention_outputs], dim=1
            )  # Shape: (batch_size, num_heads, sequence_length, sequence_length)
            return attention_output, attention_probs

        return attention_output, None

### Module API: Multi-layer Perceptron (MLP)


The `MLP` module implements a feedforward neural network used in transformers. This component processes the output of the attention mechanism, applies a non-linear transformation, and projects it back to the original dimension.


#### Key Steps:


1. Expand the input dimension using a fully connected layer.
2. Apply a non-linear activation function, such as GELU.
3. Reduce the dimension back to the input size using a second fully connected layer.
4. Apply dropout for regularization.

The input to the `MLP` module will be a tensor of shape `(batch_size, sequence_length, hidden_size)`. The output will have the same shape.


#### Implementation


**Implementation note.**

Complete the `MLP` module by:
1. Defining the two fully connected layers.
2. Adding a GELU activation function.
3. The `forward` method to:
   - Expand the input dimension.
   - Apply the non-linear activation.
   - Project the dimension back to the input size.
   - Apply dropout.

Refer to the comments in the code snippet for guidance.


In [ ]:
class MLP(nn.Module):
    """
    Implements a multi-layer perceptron (MLP) module for transformers.

    This module applies the following steps:
    1. A fully connected (dense) layer that expands the input dimension.
    2. A non-linear activation function (e.g., GELU).
    3. A second fully connected layer that projects the expanded dimension back to the input size.
    4. Dropout for regularization.

    Parameters:
    - config (dict): Configuration dictionary containing:
        - "hidden_size" (int): Dimensionality of the input and output embeddings.
        - "intermediate_size" (int): Dimensionality of the expanded hidden layer.
        - "hidden_dropout_prob" (float): Dropout probability applied after the second dense layer.
    """

    def __init__(self, config):
        super().__init__()

        self.dense1 = nn.Linear(config['hidden_size'], config['intermediate_size'])

        self.activation = nn.GELU()

        self.dense2 = nn.Linear(config['intermediate_size'], config['hidden_size'])

        # Dropout for regularization
        self.dropout = nn.Dropout(config["hidden_dropout_prob"])

    def forward(self, x):
        """
        Forward pass:
        - Input: A tensor of shape (batch_size, sequence_length, hidden_size).
        - Output: A tensor of shape (batch_size, sequence_length, hidden_size).
        """
        x = self.dense1(x)

        x = self.activation(x)

        x = self.dense2(x)

        # Apply dropout
        x = self.dropout(x)

        return x

### Module API: Transformer Block


The `Block` module implements a single **transformer block**, which combines multi-head self-attention (MHSA) and a feed-forward network (MLP), each followed by layer normalization and residual connections.


#### Key Steps:
1. Apply **Layer Normalization** before the self-attention mechanism.
2. Compute **Multi-Head Self-Attention** and add a **residual connection** (skip connection).
3. Apply **Layer Normalization** before the feed-forward network (MLP).
4. Compute the **Feed-Forward Network (MLP)** and add another **residual connection**.
5. Optionally return the **attention probabilities** for analysis or visualization.

The input to the `Block` module will be a tensor of shape `(batch_size, sequence_length, hidden_size)`. The output will have the same shape, with an optional tensor of attention probabilities.


#### Implementation


**Implementation note.**

Complete the `Block` module by:
1. Initializing the multi-head self-attention mechanism (MHSA), feed-forward network (MLP), and layer normalization layers.
2. The `forward` method to:
   - Normalize the input before self-attention and the MLP.
   - Add residual connections after self-attention and the MLP.
   - Optionally return attention probabilities.


In [ ]:
class Block(nn.Module):
    """
    Implements a single transformer block.

    A transformer block consists of:
    1. A multi-head self-attention mechanism with skip connection and layer normalization.
    2. A feed-forward network (FFN) with skip connection and layer normalization.

    Parameters:
    - config (dict): Configuration dictionary containing:
        - "hidden_size" (int): Dimensionality of the input and output embeddings.
        - "num_attention_heads" (int): Number of attention heads for the multi-head attention mechanism.
        - "qkv_bias" (bool): Whether to include biases in query, key, and value projections for attention.
        - "attention_probs_dropout_prob" (float): Dropout probability for attention probabilities.
        - "intermediate_size" (int): Dimensionality of the expanded hidden layer in the MLP.
        - "hidden_dropout_prob" (float): Dropout probability applied in the MLP and output projection.
    """

    def __init__(self, config):
        super().__init__()

        self.mhsa = MultiHeadAttention(config)

        # Initialize layer normalization before self-at tention
        self.layernorm_1 = nn.LayerNorm(config["hidden_size"])

        self.mlp = MLP(config)

        # Initialize layer normalization before the MLP
        self.layernorm_2 = nn.LayerNorm(config["hidden_size"])

    def forward(self, x, output_attentions=False):
        """
        Forward pass for a transformer block.

        Parameters:
        - x (Tensor): Input tensor of shape (batch_size, sequence_length, hidden_size).
        - output_attentions (bool): If True, returns attention probabilities.

        Returns:
        - x (Tensor): Output tensor of shape (batch_size, sequence_length, hidden_size).
        - attention_probs (Tensor, optional): Attention probabilities of shape
          (batch_size, num_heads, sequence_length, sequence_length) if `output_attentions` is True.
        """
        # Apply layer normalization before self-attention
        normalized_x = self.layernorm_1(x)

        attention_output, attention_probs = self.mhsa(normalized_x, output_attentions=output_attentions)

        x = x + attention_output

        # Apply layer normalization before the MLP
        normalized_x = self.layernorm_2(x)

        fc_output = self.mlp(normalized_x)

        x = x + fc_output

        # Return output and optionally the attention probabilities
        if output_attentions:
            return x, attention_probs
        return x, None

### Module API: Transformer Encoder


The `Encoder` module implements the **transformer encoder**, which is composed of multiple stacked transformer blocks. Each block applies multi-head self-attention (MHSA) and a multi-layer perceptron (MLP) with layer normalization and skip connections.


#### Key Steps:


1. Define multiple transformer blocks using the `Block` module.
2. Pass the input through each transformer block sequentially.
3. Optionally store and return the attention probabilities for each block.

The input to the `Encoder` module will be a tensor of shape `(batch_size, sequence_length, hidden_size)`. The output will include:
- **Final Encoder Output**: A tensor of shape `(batch_size, sequence_length, hidden_size)`.
- **Attention Probabilities** (optional): A list of tensors, each containing the attention probabilities for a block.


#### Implementation


**Implementation note.**

Complete the `Encoder` module by:
1. Initializing a stack of transformer blocks using `nn.ModuleList`.
2. The `forward` method to:
   - Pass the input through each transformer block sequentially.
   - Collect and return attention probabilities if requested.


In [ ]:
class Encoder(nn.Module):
    """
    Implements the transformer encoder module.

    The encoder consists of multiple transformer blocks stacked sequentially.
    Each block applies multi-head self-attention and a feed-forward network (MLP)
    with skip connections and layer normalization.

    Parameters:
    - config (dict): Configuration dictionary containing:
        - "hidden_size" (int): Dimensionality of the input and output embeddings.
        - "num_hidden_layers" (int): Number of transformer blocks in the encoder.
        - "num_attention_heads" (int): Number of attention heads for the multi-head attention mechanism.
        - "qkv_bias" (bool): Whether to include biases in query, key, and value projections.
        - "attention_probs_dropout_prob" (float): Dropout probability for attention probabilities.
        - "intermediate_size" (int): Dimensionality of the expanded hidden layer in the MLP.
        - "hidden_dropout_prob" (float): Dropout probability applied in the MLP and output projection.
    """

    def __init__(self, config):
        super().__init__()

        self.blocks = nn.ModuleList([Block(config) for _ in range(config['num_hidden_layers'])])

    def forward(self, x, output_attentions=False):
        """
        Forward pass for the transformer encoder.

        Parameters:
        - x (Tensor): Input tensor of shape (batch_size, sequence_length, hidden_size).
        - output_attentions (bool): If True, returns attention probabilities for each block.

        Returns:
        - x (Tensor): Output tensor of shape (batch_size, sequence_length, hidden_size).
        - all_attentions (list of Tensors, optional): List of attention probabilities, one for each block.
          Each tensor has shape (batch_size, num_heads, sequence_length, sequence_length).
        """
        # Initialize a list to store attention probabilities (if required)
        all_attentions = []

        # Store the attention probabilities of each block in `all_attentions` list     #
        for block in self.blocks:
            x, attention_probs = block(x, output_attentions=output_attentions)
            if output_attentions:
                all_attentions.append(attention_probs)


        # Return the encoder's final output and optionally the attention probabilities
        if output_attentions:
            return x, all_attentions
        return x, None

### Module API: Vision Transformer


The `VisionTransformer` module implements the complete Vision Transformer (ViT) model, designed for image classification tasks. It combines patch embeddings, a transformer encoder, and a classification head.


#### Key Components:


1. **Embedding Module**: Splits the input image into patches, adds positional encodings, and appends a [CLS] token.
2. **Transformer Encoder**: Processes the embeddings using stacked transformer blocks.
3. **Classification Head**: Projects the [CLS] token's output to logits for classification.

The input to the `VisionTransformer` module will be a tensor of shape `(batch_size, num_channels, image_size, image_size)`. The output will include:
- **Logits**: A tensor of shape `(batch_size, num_classes)` representing class scores.
- **Attention Probabilities** (optional): A list of tensors, one for each transformer block, for analysis or visualization.


#### Implementation


**Implementation note.**

Complete the `VisionTransformer` module by:
1. Initializing the embedding module, transformer encoder, and classification head.
2. The `forward` method to:
   - Compute patch embeddings with positional encodings.
   - Pass embeddings through the transformer encoder.
   - Use the [CLS] token's output for classification.
3. Adding a `_init_weights` function to initialize weights properly.


In [ ]:
class VisionTransformer(nn.Module):
    """
    Vision Transformer (ViT) model for classification tasks.

    This model consists of:
    1. An embedding module that splits the input image into patches, adds positional encodings, and appends a [CLS] token.
    2. A transformer encoder composed of multiple transformer blocks.
    3. A classification head that uses the [CLS] token's output to compute logits for classification.

    Parameters:
    - config (dict): Configuration dictionary containing:
        - "image_size" (int): Height and width of the input images (assumes square images).
        - "hidden_size" (int): Dimensionality of the transformer embeddings.
        - "num_classes" (int): Number of output classes for classification.
        - "patch_size" (int): Size of each image patch (height and width, assumes square patches).
        - "num_attention_heads" (int): Number of attention heads in the multi-head attention mechanism.
        - "num_hidden_layers" (int): Number of transformer blocks in the encoder.
        - "intermediate_size" (int): Dimensionality of the expanded hidden layer in the feed-forward network.
        - "hidden_dropout_prob" (float): Dropout probability for embeddings and MLP layers.
        - "attention_probs_dropout_prob" (float): Dropout probability for attention probabilities.
        - "qkv_bias" (bool): Whether to include biases in query, key, and value projections.
    """

    def __init__(self, config):
        super().__init__()
        # Save configuration
        self.config = config
        self.image_size = config["image_size"]
        self.hidden_size = config["hidden_size"]
        self.num_classes = config["num_classes"]

        self.embedding = Embeddings(config)

        self.encoder = Encoder(config)

        self.classifier = nn.Linear(self.hidden_size, self.num_classes)

        self._init_weights(self)

    def forward(self, x, output_attentions=False):
        """
        Forward pass of the ViT model.

        Parameters:
        - x (Tensor): Input tensor of shape (batch_size, num_channels, image_size, image_size).
        - output_attentions (bool): If True, returns attention probabilities for each encoder block.

        Returns:
        - logits (Tensor): Logits for classification, shape (batch_size, num_classes).
        - all_attentions (list of Tensors, optional): List of attention probabilities for each encoder block.
          Each tensor has shape (batch_size, num_heads, sequence_length, sequence_length).
        """
        x = self.embedding(x)

        x, all_attentions = self.encoder.forward(x, output_attentions=output_attentions)

        cls_output = x[:, 0]
        logits = self.classifier(cls_output)

        # Return logits and optionally attention probabilities
        if output_attentions:
            return logits, all_attentions
        return logits, None

    def _init_weights(self, module):
        """
        Initialize weights for the model using standard initialization techniques.
        """
        if isinstance(module, nn.Linear):
            nn.init.trunc_normal_(module.weight, std=0.02)
            if module.bias is not None:
                nn.init.zeros_(module.bias)
        elif isinstance(module, nn.LayerNorm):
            nn.init.zeros_(module.bias)
            nn.init.ones_(module.weight)

### Vision Transformer Configuration


The Vision Transformer (ViT) model requires a configuration dictionary to define its architecture and behavior.

Below, we define the configuration for our custom ViT model, which is tailored for the `CIFAR-10` dataset.

This configuration will be passed to the `VisionTransformer` class to build the model.


#### Key Configuration Parameters:


1. **Patch Size (`patch_size`)**: Determines the size of each patch. For an input image of size 32x32, a patch size of 4 results in a sequence of 8x8 = 64 patches.
2. **Hidden Size (`hidden_size`)**: The dimensionality of the embeddings and transformer layers.
3. **Number of Transformer Layers (`num_hidden_layers`)**: Specifies the number of transformer blocks in the encoder.
4. **Number of Attention Heads (`num_attention_heads`)**: Defines how many independent attention mechanisms are computed in parallel.
5. **Intermediate Size (`intermediate_size`)**: The size of the feed-forward network within each transformer block. Typically set to 4 times the `hidden_size`.
6. **Dropout Probabilities (`hidden_dropout_prob`, `attention_probs_dropout_prob`)**: Dropout rates for regularization during training.
7. **Image Size (`image_size`)**: The height and width of the input images (assumes square images).
8. **Number of Classes (`num_classes`)**: The number of output classes (e.g., 10 for CIFAR-10).
9. **Number of Channels (`num_channels`)**: The number of channels in the input images (e.g., 3 for RGB images).
10. **Query-Key-Value Bias (`qkv_bias`)**: Whether to include biases in the query, key, and value projections.
11. **Initializer Range (`initializer_range`)**: The standard deviation for weight initialization.


In [ ]:
# Configuration for custom ViT model
config = {
    "patch_size": 4,  # Input image size: 32x32 -> 8x8 patches
    "hidden_size": 48,
    "num_hidden_layers": 4,
    "num_attention_heads": 4,
    "intermediate_size": 4 * 48,  # 4 * hidden_size
    "hidden_dropout_prob": 0.0,
    "attention_probs_dropout_prob": 0.0,
    "initializer_range": 0.02,
    "image_size": 32,
    "num_classes": 10,  # CIFAR-10 number of classes
    "num_channels": 3,
    "qkv_bias": True,
    "use_faster_attention": True,
}

In [ ]:
# Check configuration validity
assert config["hidden_size"] % config["num_attention_heads"] == 0
assert config["intermediate_size"] == 4 * config["hidden_size"]
assert config["image_size"] % config["patch_size"] == 0

### Vision Transformer Trainer


The `Trainer` class manages the training and evaluation process for the Vision Transformer (ViT). It encapsulates the following functionalities:

1. **Initialization**: Accepts the model, optimizer, loss function, and device (CPU/GPU), ensuring the model is moved to the correct device.

2. **Training**: The `train_epoch` method:
   - Iterates through training batches using a `tqdm` progress bar.
   - Performs forward passes, computes loss, backpropagates, and updates model parameters.
   - Returns the average training loss for the epoch.

3. **Evaluation**: The `evaluate` method:
   - Runs the model on a validation or test set in evaluation mode (`model.eval()`).
   - Computes accuracy and average loss, displaying progress with a `tqdm` bar.

4. **Full Training Loop**: The `train` method:
   - Combines `train_epoch` and `evaluate` for multiple epochs.
   - Logs training loss, validation loss, and accuracy after each epoch.

This modular approach ensures efficient, reusable, and well-organized training and evaluation for the ViT.


#### Implementation:


In [ ]:
class Trainer:
    def __init__(self, model, optimizer, loss_fn, device):
        self.model = model.to(device)
        self.optimizer = optimizer
        self.loss_fn = loss_fn
        self.device = device

    def train_epoch(self, train_loader):
        """
        Train the model for one epoch with a progress bar.
        """
        self.model.train()
        total_loss = 0

        # Initialize tqdm progress bar
        with tqdm(train_loader, desc="Training", unit="batch") as t:
            for images, labels in t:
                images, labels = images.to(self.device), labels.to(self.device)

                # Zero gradients
                self.optimizer.zero_grad()

                # Forward pass
                logits, _ = self.model(images, True)

                # Compute loss
                loss = self.loss_fn(logits, labels)

                # Backward pass
                loss.backward()

                # Update parameters
                self.optimizer.step()

                # Accumulate loss
                total_loss += loss.item() * len(images)

                # Update progress bar
                t.set_postfix(loss=loss.item())

        # Average loss
        return total_loss / len(train_loader.dataset)

    @torch.no_grad()
    def evaluate(self, dataloader):
        """
        Evaluate the model on the given dataloader with a progress bar.
        """
        self.model.eval()
        total_loss = 0
        correct = 0

        # Initialize tqdm progress bar
        with tqdm(dataloader, desc="Validating", unit="batch") as t:
            for images, labels in t:
                images, labels = images.to(self.device), labels.to(self.device)

                # Forward pass
                logits, _ = self.model(images, True)

                # Compute loss
                loss = self.loss_fn(logits, labels)
                total_loss += loss.item() * len(images)

                # Compute accuracy
                predictions = torch.argmax(logits, dim=1)
                correct += (predictions == labels).sum().item()

        accuracy = correct / len(dataloader.dataset)
        avg_loss = total_loss / len(dataloader.dataset)

        return accuracy, avg_loss

    def train(self, train_loader, val_loader, epochs):
        """
        Train the model for the specified number of epochs.
        """
        train_losses, val_losses, accuracies = [], [], []

        for epoch in range(epochs):
            print(f"Epoch {epoch + 1}/{epochs}")

            # Train for one epoch
            train_loss = self.train_epoch(train_loader)

            # Evaluate on the val set
            accuracy, val_loss = self.evaluate(val_loader)

            # Record metrics
            train_losses.append(train_loss)
            val_losses.append(val_loss)
            accuracies.append(accuracy)

            # Print epoch metrics
            print(f"Train Loss: {train_loss:.4f}, Validation Loss: {val_loss:.4f}, Validation Accuracy: {accuracy:.4f}")

        # Return the training history
        return train_losses, val_losses, accuracies

### Training of Custom Vision Transformer


Now, it's time to train our custom Vision Transformer (ViT) model on CIFAR-10. We'll define the training parameters and utilize the `Trainer` class to handle the training and validation processes efficiently. Here's what this step entails:

1. **Training Parameters**:
   - `batch_size`: Defines the number of samples per batch during training.
   - `epochs`: The number of times the model sees the entire training dataset.
   - `learning_rate`: The step size for the optimizer when updating the model parameters.

2. **Model, Optimizer, and Loss Function**:
   - The Vision Transformer is instantiated using the defined configuration (`config`).
   - The AdamW optimizer is used, which combines the advantages of both Adam and weight decay regularization.
   - Cross-Entropy Loss is applied, as it is suitable for classification tasks.

3. **Training Process**:
   - The `Trainer` class manages the training loop, tracks losses, and computes accuracy at each epoch.


#### Implementation


##### **Implementation note.**

Complete the code below to train the custom Vision Transformer for 5 epochs. Specifically, instantiate:
- The `model` using the `VisionTransformer` class.
- The `optimizer` using `torch.optim.AdamW` with the specified learning rate and weight decay.
- The `loss_fn` as `torch.nn.CrossEntropyLoss`.

Use the `Trainer` class to train the model and track performance.


In [ ]:
# Training parameters
batch_size = 64
epochs = 5
learning_rate = 1e-2

model = VisionTransformer(config)
optimizer = torch.optim.AdamW(params=model.parameters(), lr=learning_rate)
loss_fn = torch.nn.CrossEntropyLoss()

# Instantiate the trainer
trainer = Trainer(model, optimizer, loss_fn, device)

# Train the model
train_losses, val_losses, accuracies = trainer.train(train_loader, val_loader, epochs)

### Experiment with Learning Rate


Hyperparameter tuning is a critical step in improving model performance. Let’s perform a learning rate search to see how it impacts the validation accuracy of our custom Vision Transformer.


**Implementation note.**

**Task**:

1.	Experiment with 5 to 7 different learning rates in the range around the current learning rate (1e-2), such as `[1e-3, 2e-3, 3e-3, 5e-3, 1e-2, 2e-2]`.
2.	Train the model for 5 epochs with each learning rate.
3.	Record the validation accuracy for each learning rate.
4.	Plot the learning rates (x-axis) vs. validation accuracy (y-axis).

**Note**:

- Use the Trainer class to manage training and evaluation.
- Make sure to reset the model weights and optimizer for each learning rate.
- Visualize the results to determine the best learning rate.


In [ ]:
import matplotlib.pyplot as plt

# Learning rates to test
learning_rates = [1e-3, 2e-3, 3e-3, 5e-3, 1e-2, 2e-2]
epochs = 5

# Store validation accuracies
val_accuracies = []

for lr in learning_rates:
    print(f"\nTraining with learning rate: {lr}")

    model = VisionTransformer(config)
    optimizer = torch.optim.AdamW(params=model.parameters(), lr=lr)
    loss_fn = torch.nn.CrossEntropyLoss()
    trainer = Trainer(model, optimizer, loss_fn, device)

    # Train the model and get the validation accuracy
    _, _, accuracies = trainer.train(train_loader, val_loader, epochs)
    val_accuracies.append(accuracies[-1])  # Save the final validation accuracy

# Plot the learning rates vs validation accuracies
plt.figure(figsize=(8, 5))
plt.plot(learning_rates, val_accuracies, marker='o')
plt.xscale('log')  # Use a log scale for learning rates
plt.xlabel('Learning Rate')
plt.ylabel('Validation Accuracy')
plt.title('Learning Rate Search')
plt.grid()
plt.show()

### Experiment with Patch Size


The patch size determines how the input image is divided into smaller patches, which are then embedded into a vector space. Smaller patch sizes provide more detailed representations of the image but increase the sequence length, making the model computationally heavier. Conversely, larger patch sizes reduce the sequence length but might lose fine-grained details.


**Implementation note.**

In this experiment, you will vary the patch size (`2`, `4`, and `8`) while keeping the other model configurations constant. Use the best learning rate, as found above. The goal is to:
1. Train the custom Vision Transformer model for 5 epochs using different patch sizes.
2. Record the validation accuracy for each patch size.
3. Plot the relationship between patch size and validation accuracy.

Complete the code below to perform this experiment.


In [ ]:
# Patch sizes to experiment with
patch_sizes = [2, 4, 8]

# Dictionary to store validation accuracies for each patch size
validation_accuracies = {}

for patch_size in patch_sizes:
    print(f"\nTraining with patch size: {patch_size}")

    # Update the configuration with the current patch size.
    config["patch_size"] = patch_size
    config["hidden_size"] = 48  # Ensure hidden_size remains consistent
    config["intermediate_size"] = 4 * config["hidden_size"]  # Update intermediate size

    # loss function, and a trainer instance. Use the best learning rate             #
    model = VisionTransformer(config)
    optimizer = torch.optim.AdamW(params=model.parameters(), lr=0.003)
    loss_fn = torch.nn.CrossEntropyLoss()
    trainer = Trainer(model, optimizer, loss_fn, device)

    _, _, accuracies = trainer.train(train_loader, val_loader, epochs)
    validation_accuracies[patch_size] = accuracies[-1]  # Save the final validation accuracy

# Plot the results
import matplotlib.pyplot as plt

# Plot the patch sizes (x-axis) vs. validation accuracies (y-axis).
plt.figure(figsize=(8, 6))
plt.plot(list(validation_accuracies.keys()), list(validation_accuracies.values()), marker='o')
plt.xlabel("Patch Size")
plt.ylabel("Validation Accuracy")
plt.title("Effect of Patch Size on Validation Accuracy")
plt.grid()
plt.show()

**Concept check.**

In the experiment above, smaller patch sizes resulted in higher validation accuracy. Reflect on why this outcome was expected. Consider the following:

1.	How does the size of a patch influence the level of detail captured in the input image?
2.	What is the trade-off between sequence length (resulting from smaller patch sizes) and model computational efficiency?
3.	Would you always choose the smallest possible patch size for a Vision Transformer? Why or why not?

Provide a brief explanation for your observations, considering the balance between fine-grained information and computational complexity.


**Answer.**

1.	Influence of Patch Size on Detail:
Smaller patch sizes capture more fine-grained details from the input image since each patch represents a smaller portion of the image. This provides the Vision Transformer with more information about local patterns and structures, which can lead to better learning and improved accuracy.

2.	Trade-off Between Sequence Length and Efficiency:
Decreasing the patch size increases the number of patches (sequence length), as the input image is divided into more pieces. While this helps the model capture finer details, it also increases the computational cost of self-attention, as the complexity of attention mechanisms grows quadratically with sequence length. Therefore, there is a trade-off between capturing more detail and maintaining computational efficiency.

3.	Choosing the Smallest Patch Size:
While smaller patch sizes improve accuracy by capturing finer details, they are not always ideal. Extremely small patches may lead to excessively long sequences, making the model computationally expensive and harder to optimize. Moreover, some details may not be critical for the task, so the optimal patch size balances fine-grained information with computational feasibility.

In conclusion, while smaller patch sizes generally yield better accuracy, the choice of patch size should also consider the computational resources available and the nature of the dataset.


### Experiment with Hidden Size


The hidden size defines the dimensionality of the patch embeddings and influences the capacity of the model. Larger hidden sizes can model more complex relationships but may overfit or require more data.


**Implementation note.**

Using the best learning rate and the best patch size from the previous experiments, now experiment with different hidden sizes in the Vision Transformer. The hidden size determines the dimensionality of the embeddings used in the transformer.
1.	Modify the `hidden_size` parameter in the configuration dictionary to the following values: `[16, 32, 48, 64, 96]`.
2.  Modify the `patch_size` parameter in the configuration dictionary, based on the best value found above.
3.	Train the Vision Transformer for 5 epochs with each hidden size.
4.	Record the validation accuracy for each hidden size.
5.	Plot a graph with hidden_size on the x-axis and validation accuracy on the y-axis.

Hint: Update the config dictionary with the new hidden_size values and ensure that other parameters such as intermediate_size and num_attention_heads remain valid.


In [ ]:
# Different hidden sizes to experiment with
hidden_sizes = [16, 32, 48, 64, 96]

# Store validation accuracies for each hidden size
val_accuracies = []

for hidden_size in hidden_sizes:
    print(f"Training with hidden_size = {hidden_size}")

    config['patch_size'] = 2
    config['hidden_size'] = hidden_size

    config["intermediate_size"] = 4 * hidden_size  # Ensure intermediate size scales with hidden size
    config["num_attention_heads"] = max(1, hidden_size // 16)  # Adjust attention heads for valid division


    # loss function, and a trainer instance. Use the best learning rate             #
    model = VisionTransformer(config)
    optimizer = torch.optim.AdamW(params=model.parameters(), lr=0.003)
    loss_fn = torch.nn.CrossEntropyLoss()
    trainer = Trainer(model, optimizer, loss_fn, device)

    # Train the model and evaluate validation accuracy
    _, _, accuracies = trainer.train(train_loader, val_loader, epochs=5)
    val_accuracies.append(accuracies[-1])  # Record the last validation accuracy

# Plot validation accuracy vs. hidden size
plt.figure(figsize=(8, 5))
plt.plot(hidden_sizes, val_accuracies, marker='o')
plt.title("Hidden Size vs. Validation Accuracy")
plt.xlabel("Hidden Size")
plt.ylabel("Validation Accuracy")
plt.grid(True)
plt.show()

### Experiment with Number of Hidden Layers


The number of transformer layers (hidden layers) defines the depth of the model. More layers may increase the model’s capacity but could lead to overfitting or vanishing gradients.


**Implementation note.**

Using the best learning rate, best patch size, and the best hidden size from the previous experiments, now experiment with different number of hidden layers in the Vision Transformer. The number of hidden layers determines the depth of the transformer encoder and affects both the model’s capacity and training time.

1.	Modify the `num_hidden_layers` parameter in the configuration dictionary to the following values: `[2, 4, 6]`.
2.	Train the Vision Transformer for 5 epochs with each number of hidden layers.
3.	Record the validation accuracy and the training time for each number of hidden layers.
4.	Plot two graphs:
    -	num_hidden_layers (x-axis) vs. validation accuracy (y-axis).
    -	num_hidden_layers (x-axis) vs. training time (y-axis).


In [ ]:
import time

# Define the list of hidden layer configurations to experiment with
num_hidden_layers_list = [2, 4, 6]  # Different numbers of hidden layers to try
val_accuracies = []  # Store validation accuracies for each configuration
training_times = []  # Store training times for each configuration

# Iterate over each number of hidden layers
for num_hidden_layers in num_hidden_layers_list:
    print(f"Training with num_hidden_layers = {num_hidden_layers}")

    # Also, update the configuration with the optimal `hidden_size` and `patch size` #
    config['num_hidden_layers'] = num_hidden_layers
    config['hidden_size'] = 48
    config['patch_size'] = 2

    model = VisionTransformer(config)
    optimizer = torch.optim.AdamW(params=model.parameters(), lr=0.003)
    loss_fn = torch.nn.CrossEntropyLoss()

    # Instantiate the trainer
    trainer = Trainer(model, optimizer, loss_fn, device)

    start_time = time.time()
    _, _, accuracies = trainer.train(train_loader, val_loader, epochs=5)
    end_time = time.time()
    training_time = end_time - start_time

    # Record the last validation accuracy and training time
    training_times.append(training_time)
    val_accuracies.append(accuracies[-1])  # Record the last validation accuracy

# Plot validation accuracy vs. number of hidden layers
plt.figure(figsize=(8, 5))
plt.plot(num_hidden_layers_list, val_accuracies, marker='o', label="Validation Accuracy")
plt.title("Number of Hidden Layers vs. Validation Accuracy")
plt.xlabel("Number of Hidden Layers")
plt.ylabel("Validation Accuracy")
plt.grid(True)
plt.legend()
plt.show()

# Plot training time vs. number of hidden layers
plt.figure(figsize=(8, 5))
plt.plot(num_hidden_layers_list, training_times, marker='o', color="orange", label="Training Time")
plt.title("Number of Hidden Layers vs. Training Time")
plt.xlabel("Number of Hidden Layers")
plt.ylabel("Training Time (seconds)")
plt.grid(True)
plt.legend()
plt.show()

**Concept check.**

The plots above illustrate the trade-off between the number of hidden layers in the Vision Transformer and two key metrics: validation accuracy and training time.

1.	What trends can you observe in the relationship between the number of hidden layers, validation accuracy, and training time?
2.	Why do you think validation accuracy increases with the number of hidden layers? What are the potential limitations of continuously increasing the number of layers?
3.	Based on these observations, how would you decide the optimal number of hidden layers for a practical application?


**Answer.**

1.	Trends Observed:
	-	Validation Accuracy: The validation accuracy increases as the number of hidden layers increases, which shows that deeper models capture more complex patterns but if we overdo it we might overfit the training data and observe a drop on the val_accuracy.
	-	Training Time: Training time increases linearly with the number of hidden layers because of more parameters and computations in each block.

2.	Reason for Increased Validation Accuracy:
	-	Deeper models have greater representational power, enabling them to capture and understand complex patterns.
	-	Each extra layer helps the model break down the data step by step, with deeper layers focusing on more abstract and higher-level features.

	Potential Limitations:
	-	Adding more layers eventually stops making a big difference in accuracy because the model might become more powerful than needed for the task.
	-	A very deep model might overfit the training data, leading to reduced generalization on the validation or test set, as seen on the plot above (between layers size 4 and 6). Deeper models are also harder to train due to issues like vanishing gradients.

3.	Deciding the Optimal Number of Hidden Layers:
	-	Essentialy it's a trade-off between accuracy and resources (training time, computational cost etc).
	-	Look for the point where the validation accuracy starts to plateau relative to the increase in training time.
	-	For online apps with constrained resources we might prefer faster training/inference over small improvements in accuracy (e.g. IoT devices)


### Experiment with Number of Attention Heads


The number of attention heads affects how the model learns to focus on different parts of the sequence. More heads allow more granular attention patterns but may not scale well with smaller hidden sizes.


**Implementation note.**

Using the best learning rate, best patch size, and the best hidden size and number of hidden layers from the previous experiments, now experiment with different number of attention heads in the Vision Transformer. The number of attention heads affects how the model distributes attention across the input sequence.

1.	Modify the `num_attention_heads` parameter in the configuration dictionary to the following values: `[2, 4, 8]`.
2.	Train the Vision Transformer for 5 epochs with each number of attention heads.
3.	Record the validation accuracy for each configuration.
4.	Plot a graph of num_attention_heads (x-axis) vs. validation accuracy (y-axis).


In [ ]:
# Define the list of attention head configurations to experiment with
num_attention_heads_list = [2, 4, 8]  # Different numbers of attention heads to try
val_accuracies = []  # Store validation accuracies for each configuration

# Iterate over each number of attention heads
for num_attention_heads in num_attention_heads_list:
    print(f"Training with num_attention_heads = {num_attention_heads}")

    # Also, update with the optimal `hidden_size`, `patch size`, num_hidden_layers #
    config['num_hidden_layers'] = 4
    config['hidden_size'] = 48
    config['patch_size'] = 2
    config['num_attention_heads'] = num_attention_heads

    # Ensure that the hidden size is divisible by the number of attention heads
    assert config["hidden_size"] % config["num_attention_heads"] == 0, \
        "hidden_size must be divisible by num_attention_heads"

    model = VisionTransformer(config)
    optimizer = torch.optim.AdamW(params=model.parameters(), lr=0.003)
    loss_fn = torch.nn.CrossEntropyLoss()

    # Instantiate the trainer
    trainer = Trainer(model, optimizer, loss_fn, device)

    # Train the model for 5 epochs and record the final validation accuracy
    _, _, accuracies = trainer.train(train_loader, val_loader, epochs=5)
    val_accuracies.append(accuracies[-1])

# Plot validation accuracy vs. number of attention heads
plt.figure(figsize=(8, 5))
plt.plot(num_attention_heads_list, val_accuracies, marker='o', label="Validation Accuracy")
plt.title("Number of Attention Heads vs. Validation Accuracy")
plt.xlabel("Number of Attention Heads")
plt.ylabel("Validation Accuracy")
plt.grid(True)
plt.legend()
plt.show()

### Visualize Attention Maps


Attention maps are a key feature of Vision Transformers, as they help us understand which parts of an image the model is focusing on while making predictions.

These maps represent how much “attention” the model is paying to different image regions.

By visualizing attention maps, we can gain insights into the interpretability of the model and verify if it is focusing on relevant parts of the image.


**Implementation note.**

In this notebook, you have to visualize attention maps for the Vision Transformer model trained with the best configuration. Specifically, you have to:

- Forward pass through the model and extract attention maps.
- Select `num_per_class` images and attention maps for visualization.
- Plot both the images and their corresponding attention maps.

The goal is to observe how the model attends to different regions in images for various classes.


In [ ]:
import torch
import matplotlib.pyplot as plt

def visualize_attention_maps(model, dataloader, num_per_class=2, device='cuda'):
    """
    Visualize attention maps for `num_per_class` images per class.

    Parameters:
    - model: Trained Vision Transformer model.
    - dataloader: DataLoader for the dataset.
    - num_per_class: Number of images to visualize per class.
    - device: The device to run the model on (default: 'cuda').
    """
    model.eval()  # Set the model to evaluation mode
    images_per_class = {i: 0 for i in range(10)}  # Track images per class
    images, attention_maps, classes = [], [], []

    with torch.no_grad():
        for images_batch, labels_batch in dataloader:
            images_batch, labels_batch = images_batch.to(device), labels_batch.to(device)

            outputs, attnt_maps = model.forward(images_batch, output_attentions=True)


            for img, label, attention_map_per_layer in zip(images_batch, labels_batch, attnt_maps):
                if images_per_class[label.item()] < num_per_class:
                    images_per_class[label.item()] += 1
                    images.append(img.cpu())


                    avg_attn_map = attention_map_per_layer.mean(dim=1)  # Average over heads (shape: [batch_size, seq_len, seq_len])
                    attention_maps.append(avg_attn_map[0].cpu())  # Take the first image in the batch

                    classes.append(label.cpu())

                # Stop collecting if we have enough images for all classes
                if all(count >= num_per_class for count in images_per_class.values()):
                    break

            if all(count >= num_per_class for count in images_per_class.values()):
                break

    # Normalize attention maps to [0, 1]
    attention_maps = [am / am.max() for am in attention_maps]

    # Define grid for visualization: 10 rows (classes) and 2 columns per image
    num_rows = 10
    num_cols = num_per_class * 2

    fig, axs = plt.subplots(num_rows, num_cols, figsize=(num_cols * 2, num_rows * 2))

    for idx, (image, attn_map, cls) in enumerate(zip(images, attention_maps, classes)):
        row = cls
        col = (idx % num_per_class) * 2

        # Visualize the original image
        img = image.permute(1, 2, 0)  # Convert (C, H, W) to (H, W, C)
        img = (img * 0.5 + 0.5).clamp(0, 1).numpy()  # De-normalize and clip to [0, 1]
        axs[row, col].imshow(img)
        axs[row, col].axis("off")
        axs[row, col].set_title(f"Class {cls.item()}")  # Display class name or index

        # Visualize the attention map
        axs[row, col + 1].imshow(attn_map, cmap="cividis")  # Purple-yellow colormap
        axs[row, col + 1].axis("off")
        axs[row, col + 1].set_title("Attention Map")

    plt.tight_layout()
    plt.show()

In [ ]:
# Pass the validation DataLoader to visualize attention maps
visualize_attention_maps(model, val_loader, num_per_class=4)

Advanced data augmentation (e.g., CutMix, MixUp, RandAugment) can improve generalization and make the model more robust to variations in the input data.


## Fine-tune a pre-trained ViT-Tiny (ViT-T/16)


Fine-tuning a pre-trained Vision Transformer (ViT) involves taking a model that has already been trained on a large dataset (e.g., ImageNet-1k) and adapting it to a new, smaller dataset (e.g., CIFAR-10).

The benefit is that the model already has learned features that can be fine-tuned to the new task, saving time and computational resources compared to training from scratch.


### Install `timm` library


`timm` is a PyTorch library that provides access to state-of-the-art pre-trained models, including Vision Transformers. You need to install it before importing the pre-trained ViT-Tiny.


#### Modify `Trainer`


We have to slightly modify our `Trainer`:


In [ ]:
class Trainer:
    def __init__(self, model, optimizer, loss_fn, device):
        self.model = model.to(device)
        self.optimizer = optimizer
        self.loss_fn = loss_fn
        self.device = device

    def train_epoch(self, train_loader):
        """
        Train the model for one epoch with a progress bar.
        """
        self.model.train()
        total_loss = 0

        # Initialize tqdm progress bar
        with tqdm(train_loader, desc="Training", unit="batch") as t:
            for images, labels in t:
                images, labels = images.to(self.device), labels.to(self.device)

                # Zero gradients
                self.optimizer.zero_grad()

                # Forward pass
                logits = self.model(images)

                # Compute loss
                loss = self.loss_fn(logits, labels)

                # Backward pass
                loss.backward()

                # Update parameters
                self.optimizer.step()

                # Accumulate loss
                total_loss += loss.item() * len(images)

                # Update progress bar
                t.set_postfix(loss=loss.item())

        # Average loss
        return total_loss / len(train_loader.dataset)

    @torch.no_grad()
    def evaluate(self, dataloader):
        """
        Evaluate the model on the given dataloader with a progress bar.
        """
        self.model.eval()
        total_loss = 0
        correct = 0

        # Initialize tqdm progress bar
        with tqdm(dataloader, desc="Validating", unit="batch") as t:
            for images, labels in t:
                images, labels = images.to(self.device), labels.to(self.device)

                # Forward pass
                logits = self.model(images)

                # Compute loss
                loss = self.loss_fn(logits, labels)
                total_loss += loss.item() * len(images)

                # Compute accuracy
                predictions = torch.argmax(logits, dim=1)
                correct += (predictions == labels).sum().item()

        accuracy = correct / len(dataloader.dataset)
        avg_loss = total_loss / len(dataloader.dataset)

        return accuracy, avg_loss

    def train(self, train_loader, val_loader, epochs):
        """
        Train the model for the specified number of epochs.
        """
        train_losses, val_losses, accuracies = [], [], []

        for epoch in range(epochs):
            print(f"Epoch {epoch + 1}/{epochs}")

            # Train for one epoch
            train_loss = self.train_epoch(train_loader)

            # Evaluate on the val set
            accuracy, val_loss = self.evaluate(val_loader)

            # Record metrics
            train_losses.append(train_loss)
            val_losses.append(val_loss)
            accuracies.append(accuracy)

            # Print epoch metrics
            print(f"Train Loss: {train_loss:.4f}, Validation Loss: {val_loss:.4f}, Validation Accuracy: {accuracy:.4f}")

        # Return the training history
        return train_losses, val_losses, accuracies

### Implementation


**Implementation note.**

Fill in the missing parts of the following code to complete the fine-tuning process. Key steps:

1.	Preprocessing the CIFAR-10 Dataset:
	-	CIFAR-10 images are  32 $\times$ 32 , while the pre-trained ViT-Tiny expects images of  224 $\times$ 224 . We resize the images during preprocessing.
	-	Normalization is applied using the CIFAR-10 dataset’s mean and standard deviation.
2.	Loading the Pre-trained ViT-Tiny Model:
	-	The model is loaded with pre-trained weights on ImageNet-1k.
	-	The classifier head is modified to match the number of classes in CIFAR-10 (10 classes).
3.	Fine-tuning the Model:
	-	The model’s parameters are fine-tuned on the CIFAR-10 dataset using the AdamW optimizer and cross-entropy loss.
	-	Training is performed for 5 epochs, and validation accuracy is recorded after each epoch.
4.	Plotting Validation Accuracy:
	-	A plot is created to show how validation accuracy evolves over epochs.


In [ ]:
# Import necessary libraries
from timm import create_model
from torch.utils.data import DataLoader, random_split
from torchvision.datasets import CIFAR10

# Define the mean and standard deviation for CIFAR-10
mean = (0.4914, 0.4822, 0.4465)  # Precomputed mean for CIFAR-10
std = (0.2023, 0.1994, 0.2010)   # Precomputed std for CIFAR-10

# Define dataset path
dataset_path = "data"

# images to 224x224, including normalization using the provided `mean` and `std`. #
transform_resized = transforms.Compose([
    transforms.Resize(224),                    # Resize to 224x224
    transforms.ToTensor(),                      # Convert image to tensor
    transforms.Normalize(mean, std)             # Normalize using CIFAR-10 mean and std
])

# Use the same number of images for train, val, and test splits
num_train = 45000  # Number of training examples
num_val = 5000    # Number of validation examples
num_test = 10000   # Number of test examples (CIFAR-10 test set size)

# Use the splits defined above to define all the subsets with the resized images.      #
cifar10_data = CIFAR10(root=dataset_path, train=True, download=True, transform=transform_resized)
train_data, val_data = random_split(cifar10_data, [num_train, num_val])
test_data = CIFAR10(root=dataset_path, train=False, download=True, transform=transform_resized)


# Use the same batch size as before
batch_size = 64

train_loader_resized = DataLoader(train_data, batch_size=batch_size, shuffle=True)
val_loader_resized = DataLoader(val_data, batch_size=batch_size, shuffle=False)
test_loader_resized = DataLoader(test_data, batch_size=batch_size, shuffle=False)

# Load the pre-trained ViT-Tiny model from timm
print("Loading pre-trained ViT-Tiny model...")
model = create_model(
    "vit_tiny_patch16_224",  # Pre-trained ViT-Tiny on ImageNet-1k
    pretrained=True,        # Load pre-trained weights
    num_classes=10          # Adapt classifier head to CIFAR-10 (10 classes)
)

# Define optimizer and loss function
optimizer = optim.AdamW(model.parameters(), lr=2e-4, weight_decay=1e-2)
loss_fn = nn.CrossEntropyLoss()

# Instantiate the trainer
trainer = Trainer(model, optimizer, loss_fn, device)

# Train the model for 5 epochs
print("Starting fine-tuning...")
train_losses, val_losses, accuracies = trainer.train(train_loader_resized, val_loader_resized, epochs=5)

# Plot validation accuracy over epochs
plt.figure(figsize=(8, 5))
plt.plot(range(1, len(accuracies) + 1), accuracies, marker='o', label="Validation Accuracy")
plt.title("Validation Accuracy over Epochs")
plt.xlabel("Epoch")
plt.ylabel("Validation Accuracy")
plt.grid(True)
plt.legend()
plt.show()

## Fine-tune only the classifier of a ViT-Tiny (ViT-T/16)


Now, we will leverage the same pre-trained model from the `timm` library, but instead of fine-tuning the entire model, we freeze all layers except the classifier head.

This approach is computationally efficient and allows us to adapt the powerful features learned during pre-training to our target dataset, CIFAR-10.


#### Implementation


**Implementation note.**

Complete the missing parts in the code to fine-tune the classifier of a pre-trained ViT-T/16 model on CIFAR-10. Key steps:

1.	**Loading the Pre-trained Model**: We load the ViT-T/16 model pre-trained on ImageNet-1k and modify its classifier head to handle CIFAR-10’s 10 classes.
2.	**Freezing Layers**: All layers are frozen except the classifier. This ensures that the pre-trained feature extractor is not altered during training.
3.	**Fine-tuning the Classifier**: The optimizer is applied only to the classifier, reducing training complexity and time.


In [ ]:
print("Loading pre-trained ViT-Tiny model...")
model = create_model(
    "vit_tiny_patch16_224",  # Pre-trained ViT-Tiny on ImageNet-1k
    pretrained=True,        # Load pre-trained weights
    num_classes=10          # Adapt classifier head to CIFAR-10 (10 classes)
)

# Freeze all layers except the classifier
for param in model.parameters():
    param.requires_grad = False

# Unfreeze the classifier layer
for param in model.head.parameters():
    param.requires_grad = True

# Pass only the head/classifier parameters to the optimizer.                   #
optimizer = optim.AdamW(model.head.parameters(), lr=2e-4, weight_decay=1e-2)
loss_fn = nn.CrossEntropyLoss()

# Instantiate the trainer
trainer = Trainer(model, optimizer, loss_fn, device)

# Train the model for 5 epochs
train_losses, val_losses, accuracies = trainer.train(train_loader_resized, val_loader_resized, epochs=5)

# Plot validation accuracy over epochs
plt.figure(figsize=(8, 5))
plt.plot(range(1, len(accuracies) + 1), accuracies, marker='o', label="Validation Accuracy")
plt.title("Validation Accuracy over Epochs")
plt.xlabel("Epoch")
plt.ylabel("Validation Accuracy")
plt.grid(True)
plt.legend()
plt.show()

## Fine-tune only the classifier of a ViT-Base (ViT-B/16)


Now, we will leverage a larger pre-trained model, the ViT-B/16 from the `timm` library, which has been trained on ImageNet-1k.

We will freeze all layers except the classifier head. We will again fine-tune only the classifier.

As we've said, this approach is computationally efficient and allows us to adapt the powerful features learned during pre-training to our target dataset, CIFAR-10.


### Implementation


**Implementation note.**

Complete the missing parts in the code to fine-tune the classifier of a pre-trained ViT-B/16 model for CIFAR-10. Key steps:

1.	**Loading the Pre-trained Model**: We load the ViT-B/16 model pre-trained on ImageNet-1k and modify its classifier head to handle CIFAR-10’s 10 classes.
2.	**Freezing Layers**: All layers are frozen except the classifier. This ensures that the pre-trained feature extractor is not altered during training.
3.	**Fine-tuning the Classifier**: The optimizer is applied only to the classifier, reducing training complexity and time.


In [ ]:
print("Loading pre-trained ViT-Tiny model...")
model = create_model(
    "vit_base_patch16_224",  # Pre-trained ViT-Base on ImageNet-1k
    pretrained=True,        # Load pre-trained weights
    num_classes=10          # Adapt classifier head to CIFAR-10 (10 classes)
)

# Freeze all layers except the classifier
for param in model.parameters():
    param.requires_grad = False

# Unfreeze the classifier layer
for param in model.head.parameters():
    param.requires_grad = True

# Pass only the head/classifier parameters to the optimizer.                   #
optimizer = optim.AdamW(model.head.parameters(), lr=2e-4, weight_decay=1e-2)
loss_fn = nn.CrossEntropyLoss()

# Instantiate the trainer
trainer = Trainer(model, optimizer, loss_fn, device)

# Train the model for 5 epochs
train_losses, val_losses, accuracies = trainer.train(train_loader_resized, val_loader_resized, epochs=5)

# Plot validation accuracy over epochs
plt.figure(figsize=(8, 5))
plt.plot(range(1, len(accuracies) + 1), accuracies, marker='o', label="Validation Accuracy")
plt.title("Validation Accuracy over Epochs")
plt.xlabel("Epoch")
plt.ylabel("Validation Accuracy")
plt.grid(True)
plt.legend()
plt.show()

## Evaluate the best model on the test set of CIFAR-10


Having done all these different experiments, evaluating smaller and bigger, custom and well-known models on the validation set, now it is time to evaluate the best model on the the CIFAR-10 test set.

Below we evaluate the fine-tuned (only the classifier) ViT-Base model.


In [ ]:
test_acc, test_loss = trainer.evaluate(test_loader_resized)

In [ ]:
print(f'Accuracy on CIFAR-10 test set: {test_acc*100}%')

## Technical Report


**Concept check.**

Write a technical report summarizing your observations and conclusions from all the experiments conducted in this notebook. Your report should address the following points:
1.	Custom Vision Transformer:
	-	What impact did hyperparameter tuning (learning rate, patch size, hidden size, number of layers, attention heads) have on validation accuracy and training time?
	-	What trade-offs were observed between model complexity and performance?
2.	Visualization of Attention Maps:
	-	What insights did you gain from visualizing attention maps?
	-	How do these visualizations help in understanding the model’s focus on input images?
3.	Fine-tuning Pre-trained Vision Transformers:
	-	How did fine-tuning the entire ViT-Tiny model perform compared to your custom Vision Transformer?
	-	How did fine-tuning the entire ViT-Tiny model perform compared to fine-tuning only the classifier of the same model?
	-	How does training only the classifier of ViT-Base compare to fine-tuning the entire ViT-Tiny?
4.	General Observations:
	-	Which approach (custom ViT or pre-trained models) seemed most effective for CIFAR-10?
	-	Discuss the computational efficiency and validation accuracy trade-offs observed in this notebook.


**Impact of Hyperparameter Tuning on Validation Accuracy and Training Time**

**Learning Rate:**

- Lower learning rates led to slower convergence but better stability and higher final validation accuracy.

- Higher learning rates accelerated convergence but sometimes caused instability, leading to suboptimal accuracy or divergence.

**Patch Size:**

- Smaller patch sizes allowed the model to capture finer-grained details, improving validation accuracy. However, this increased computational cost and training time due to larger input sequence lengths.

- Larger patch sizes reduced the sequence length, improving training speed but at the expense of accuracy as finer details were lost.

**Hidden Size:**

- Increasing the hidden size improved the model’s capacity to learn complex patterns, resulting in higher accuracy. However, it also significantly increased training time and memory consumption.

**Number of Layers:**

- Deeper models (more layers) generally yielded better validation accuracy due to their higher representational capacity but suffered from increased training time and potential overfitting on limited data.

**Attention Heads:**

- Increasing the number of attention heads improved the model’s ability to focus on diverse aspects of the input, leading to better accuracy. However, this came at the cost of higher computational demands and longer training time.

**Trade-offs Between Model Complexity and Performance**

Higher complexity (e.g., more layers, larger hidden sizes, smaller patch sizes) improved accuracy but significantly increased computational cost and training time.

Simpler models trained faster and were more resource-efficient but often underperformed in terms of validation accuracy, especially on complex patterns.


**Visualization of Attention Maps**


**Insights Gained**

Attention maps didn't reveal much, maybe a faulty plotting code is responsible for this.


**Fine-tuning Pre-trained Vision Transformers**

**Performance of Fine-tuning the Entire ViT-Tiny Model vs. Custom Vision Transformer**

Fine-tuning the entire pre-trained ViT-Tiny model consistently outperformed the custom Vision Transformer in terms of validation accuracy.

The pre-trained ViT-Tiny benefited from transfer learning.

**Performance of Fine-tuning the Entire ViT-Tiny Model vs. Fine-tuning Only the Classifier**

Fine-tuning the entire ViT-Tiny model achieved higher validation accuracy than fine-tuning only the classifier.

Fine-tuning the entire model allowed it to adapt its internal representations to the CIFAR-10 dataset, whereas tuning only the classifier limited adaptation to the final layers.

**Training Only the Classifier of ViT-Base vs. Fine-tuning the Entire ViT-Tiny**

Training only the classifier of ViT-Base (a larger model) performed worse than fine-tuning the entire ViT-Tiny.

This highlights the importance of adapting the entire model to the target dataset rather than relying solely on pre-trained representations in the classifier layer.


**General Observations**

**Effectiveness for CIFAR-10**

Pre-trained models (e.g., ViT-Tiny and ViT-Base) were more effective than the custom Vision Transformer in terms of both accuracy and training efficiency.

The custom Vision Transformer required extensive hyperparameter tuning to achieve good performance, while pre-trained models provided strong results with less effort.

**Computational Efficiency and Validation Accuracy Trade-offs**

Fine-tuning pre-trained models was computationally more efficient and achieved better accuracy compared to training custom Vision Transformers from scratch.

Custom Vision Transformers, while flexible, demanded significant resources and big sized datasets to explore hyperparameter spaces and achieve optimal performance.

Training only the classifier was the most computationally efficient approach but resulted in lower accuracy compared to fine-tuning the entire model.
